<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/big_C_cloudflare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
# Install necessary packages for Selenium in Colab
!apt-get update
!apt-get install -y chromium-browser chromium-chromedriver
!pip install selenium bs4 pandas
!pip install playwright beautifulsoup4 polars
!playwright install chromium
!playwright install-deps

# 1. Clear out the broken Ubuntu packages
!apt-get remove -y chromium-browser chromium-chromedriver

# 2. Download and install the official Google Chrome stable version
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
!apt-get -y update
!apt-get install -y google-chrome-stable

# 3. Install Python dependencies including the auto-installer
!pip install selenium bs4 pandas chromedriver-autoinstaller polars xlsxwriter fastexcel curl_cffi playwright beautifulsoup4 polars

!playwright install
!playwright install-deps


%%capture
# 1. Install all dependencies including pandas
!pip install xlsxwriter scrapling patchright msgspec browserforge nest_asyncio polars -q
!patchright install chromium
!patchright install-deps

In [2]:
%%capture
# 1. Install all dependencies including pandas
!pip install playwright xlsxwriter scrapling patchright msgspec browserforge nest_asyncio polars -q
!patchright install chromium
!patchright install-deps

In [3]:
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import polars as pl
import asyncio
import xlsxwriter
import datetime
from datetime import date
import IPython.display as display
import datetime
today_date = datetime.datetime.now().strftime("%Y-%m-%d")
print("Today is",today_date)

Today is 2026-04-09


In [6]:
%%script echo skipping
# @title work
# This code works
import asyncio
import nest_asyncio
import pandas as pd
from scrapling.fetchers import StealthyFetcher

nest_asyncio.apply()

async def scrape_bigc_multi_pages():
    base_url = "https://www.bigc.co.th/en/category/laundry?brand=184%2C249%2C188%2C189%2C185"
    current_page = 2
    all_data = []

    # กำหนดค่าภาษาอังกฤษ
    en_cookies = [
        {'name': 'language', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'},
        {'name': 'NEXT_LOCALE', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'}
    ]
    en_headers = {'Accept-Language': 'en-US,en;q=0.9'}

    print("เริ่มการดึงข้อมูลหลายหน้า...")

    while True:
        target_url = f"{base_url}&page={current_page}"
        print(f"กำลังดึงข้อมูลหน้า {current_page}: {target_url}")

        try:
            page = await StealthyFetcher.async_fetch(
                target_url,
                headless=True,
                solve_cloudflare=True,
                cookies=en_cookies,
                headers=en_headers,
                timeout=60000,
                network_idle=True
            )

            if page.status == 200:
                containers = page.css('div[class*="productCard_container"]')

                # ถ้าไม่พบสินค้าในหน้านี้ แสดงว่าถึงหน้าสุดท้ายแล้ว
                if not containers:
                    print(f"ไม่พบสินค้าเพิ่มเติมที่หน้า {current_page} จบการทำงาน")
                    break

                print(f"พบสินค้า {len(containers)} รายการในหน้า {current_page}")

                for item in containers:
                    name = item.css('div[class*="productCard_title"] a::text').get()
                    sale_price = item.css('span[class*="productCard_sale_price"]::text').get()
                    original_price = item.css('div[class*="productCard_base_price"]::text').get()

                    all_data.append({
                        "page": current_page,
                        "product_name": name.strip() if name else "N/A",
                        "sale_price": sale_price.strip() if sale_price else "N/A",
                        "original_price": original_price.strip().replace('฿', '') if original_price else "N/A"
                    })

                # เพิ่มเลขหน้าเพื่อไปหน้าถัดไป
                current_page += 1

                # ใส่หน่วงเวลาเล็กน้อยเพื่อไม่ให้โดนบล็อก
                await asyncio.sleep(2)
            else:
                print(f"เกิดข้อผิดพลาดที่หน้า {current_page} (Status: {page.status}) หยุดการทำงาน")
                break

        except Exception as e:
            print(f"เกิดข้อผิดพลาดที่หน้า {current_page}: {e}")
            break

    # สรุปผลลัพธ์
    if all_data:
        df = pd.DataFrame(all_data)
        print(f"\nดึงข้อมูลสำเร็จทั้งหมด {len(df)} รายการ จากหน้า 2 ถึง {current_page-1}")
        display(df)
        df.to_csv('bigc_laundry_full.csv', index=False)
        print("บันทึกข้อมูลลงไฟล์ bigc_laundry_full.csv เรียบร้อยแล้ว")
    else:
        print("ไม่พบข้อมูลที่จะบันทึก")

# เริ่มทำงาน
await scrape_bigc_multi_pages()

skipping


In [7]:
# @title UDF scrape big C
# 1. ติดตั้งไลบรารีที่จำเป็น (รวมถึง polars)
# !pip install scrapling patchright msgspec browserforge nest_asyncio polars -q
# !patchright install chromium
# !patchright install-deps

import asyncio
import nest_asyncio
import polars as pl
from scrapling.fetchers import StealthyFetcher

nest_asyncio.apply()

async def get_scrape_bigc_multi_pages_polars(base_url):
    current_page = 1
    all_data = []

    # กำหนดค่าคุกกี้และ headers เพื่อรักษาเซสชันภาษาอังกฤษ
    en_cookies = [
        {'name': 'language', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'},
        {'name': 'NEXT_LOCALE', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'}
    ]
    en_headers = {'Accept-Language': 'en-US,en;q=0.9'}

    print("เริ่มการดึงข้อมูลหลายหน้าด้วย Polars...")

    while True:
        # ตรวจสอบโครงสร้าง URL เพื่อต่อพารามิเตอร์ page ให้ถูกต้อง
        separator = "&" if "?" in base_url else "?"
        target_url = f"{base_url}{separator}page={current_page}"
        print(f"กำลังดึงข้อมูลหน้า {current_page}: {target_url}")

        try:
            # ใช้ StealthyFetcher เพื่อข้าม Cloudflare WAF
            page = await StealthyFetcher.async_fetch(
                target_url,
                headless=True,
                solve_cloudflare=True, # จัดการกับระบบตรวจสอบบอทอัตโนมัติ
                cookies=en_cookies,
                headers=en_headers,
                timeout=60000,
                network_idle=True
            )

            if page.status == 200:
                containers = page.css('div[class*="productCard_container"]')

                # หากไม่พบสินค้าในหน้านี้ แสดงว่าถึงหน้าสุดท้ายแล้ว
                if not containers:
                    print(f"ไม่พบสินค้าเพิ่มเติมที่หน้า {current_page} จบการทำงาน")
                    break

                print(f"พบสินค้า {len(containers)} รายการในหน้า {current_page}")

                for item in containers:
                    # สกัดข้อมูลชื่อสินค้าและราคาจาก DOM
                    name = item.css('div[class*="productCard_title"] a::text').get()
                    sale_price = item.css('span[class*="productCard_sale_price"]::text').get()
                    original_price = item.css('div[class*="productCard_base_price"]::text').get()

                    all_data.append({
                        "page": current_page,
                        "product_name": name.strip() if name else "N/A",
                        "sale_price": sale_price.strip() if sale_price else "N/A",
                        "original_price": original_price.strip().replace('฿', '') if original_price else "N/A"
                    })

                current_page += 1
                # หน่วงเวลาเล็กน้อยเพื่อหลีกเลี่ยงการถูกตรวจจับพฤติกรรม
                await asyncio.sleep(2)
            else:
                print(f"เกิดข้อผิดพลาดที่หน้า {current_page} (Status: {page.status}) หยุดการทำงาน")
                break

        except Exception as e:
            print(f"เกิดข้อผิดพลาดที่หน้า {current_page}: {e}")
            break

    # สรุปผลลัพธ์และส่งคืนค่าเป็น Polars DataFrame
    if all_data:
        df = pl.DataFrame(all_data)
        print(f"\nดึงข้อมูลสำเร็จทั้งหมด {len(df)} รายการ จากหน้า 1 ถึง {current_page-1}")
        return df
    else:
        print("ไม่พบข้อมูลที่จะประมวลผล")
        return pl.DataFrame()

In [8]:
# @title laundry
# เริ่มทำงานและเก็บผลลัพธ์ลงในตัวแปร
df_big_c_laundry = await get_scrape_bigc_multi_pages_polars(base_url="https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100")
# แสดงตัวอย่างข้อมูล
if not df_big_c_laundry.is_empty():
    print(df_big_c_laundry.head(10))

เริ่มการดึงข้อมูลหลายหน้าด้วย Polars...
กำลังดึงข้อมูลหน้า 1: https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=1


[2026-04-09 08:08:05] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-04-09 08:09:05] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=1> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=1> (referer: https://www.google.com/)


พบสินค้า 99 รายการในหน้า 1
กำลังดึงข้อมูลหน้า 2: https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=2


[2026-04-09 08:10:18] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-04-09 08:11:19] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=2> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=2> (referer: https://www.google.com/)


พบสินค้า 99 รายการในหน้า 2
กำลังดึงข้อมูลหน้า 3: https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=3


...

พบสินค้า 96 รายการในหน้า 3
กำลังดึงข้อมูลหน้า 4: https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=4


[2026-04-09 08:13:48] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-04-09 08:13:48] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=4> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=4> (referer: https://www.google.com/)


พบสินค้า 42 รายการในหน้า 4
กำลังดึงข้อมูลหน้า 5: https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=5


[2026-04-09 08:13:59] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-04-09 08:13:59] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=5> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=5> (referer: https://www.google.com/)


ไม่พบสินค้าเพิ่มเติมที่หน้า 5 จบการทำงาน

ดึงข้อมูลสำเร็จทั้งหมด 336 รายการ จากหน้า 1 ถึง 4
shape: (10, 4)
┌──────┬─────────────────────────────────┬────────────┬────────────────┐
│ page ┆ product_name                    ┆ sale_price ┆ original_price │
│ ---  ┆ ---                             ┆ ---        ┆ ---            │
│ i64  ┆ str                             ┆ str        ┆ str            │
╞══════╪═════════════════════════════════╪════════════╪════════════════╡
│ 1    ┆ FINELINE Fabric Softener Sunsh… ┆ 70.00      ┆ N/A            │
│ 1    ┆ FINELINE Fabric Softener Gentl… ┆ 70.00      ┆ N/A            │
│ 1    ┆ FINELINE Premium Clean Cleary … ┆ 179.00     ┆ N/A            │
│ 1    ┆ FINELINE Fabric Softener Pink … ┆ 70.00      ┆ N/A            │
│ 1    ┆ FINELINE Fabric Softener Red R… ┆ 70.00      ┆ N/A            │
│ 1    ┆ FINELINE Plus Liquid Laundry D… ┆ 179.00     ┆ N/A            │
│ 1    ┆ FINELINE Organic Aloe Vera Liq… ┆ 179.00     ┆ N/A            │
│ 1    ┆ FINELINE

In [9]:
# @title dishwash
# เริ่มทำงานและเก็บผลลัพธ์ลงในตัวแปร
df_big_c_dishwash = await get_scrape_bigc_multi_pages_polars(base_url="https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100")
# แสดงตัวอย่างข้อมูล
if not df_big_c_dishwash.is_empty():
    print(df_big_c_dishwash.head(10))

เริ่มการดึงข้อมูลหลายหน้าด้วย Polars...
กำลังดึงข้อมูลหน้า 1: https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=1


[2026-04-09 08:15:12] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-04-09 08:16:13] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=1> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=1> (referer: https://www.google.com/)


พบสินค้า 90 รายการในหน้า 1
กำลังดึงข้อมูลหน้า 2: https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=2


[2026-04-09 08:16:23] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-04-09 08:16:24] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=2> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=2> (referer: https://www.google.com/)


พบสินค้า 3 รายการในหน้า 2
กำลังดึงข้อมูลหน้า 3: https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=3


[2026-04-09 08:16:33] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-04-09 08:16:34] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=3> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=3> (referer: https://www.google.com/)


ไม่พบสินค้าเพิ่มเติมที่หน้า 3 จบการทำงาน

ดึงข้อมูลสำเร็จทั้งหมด 93 รายการ จากหน้า 1 ถึง 2
shape: (10, 4)
┌──────┬─────────────────────────────────┬────────────┬────────────────┐
│ page ┆ product_name                    ┆ sale_price ┆ original_price │
│ ---  ┆ ---                             ┆ ---        ┆ ---            │
│ i64  ┆ str                             ┆ str        ┆ str            │
╞══════╪═════════════════════════════════╪════════════╪════════════════╡
│ 1    ┆ SUNLIGHT Lemon Turbo Dishwashi… ┆ 99.00      ┆ 130.00         │
│ 1    ┆ PRO Dishwashing liquid Intense… ┆ 9.00       ┆ 10.50          │
│ 1    ┆ LIPON F JAPANESE PEACH Dish Wa… ┆ 25.00      ┆ 29.00          │
│ 1    ┆ SUNLIGHT Lemon Turbo Dishwashi… ┆ 159.00     ┆ 179.00         │
│ 1    ┆ LIPON F Concentrated Dishwashi… ┆ 25.00      ┆ 29.00          │
│ 1    ┆ SUNLIGHT Lemon Turbo Dishwashi… ┆ 23.00      ┆ 32.00          │
│ 1    ┆ Lipon F Dishwashing Liquid Kaf… ┆ 25.00      ┆ 29.00          │
│ 1    ┆ SUNLIGHT 

In [10]:
list_big_c = [df_big_c_laundry, df_big_c_dishwash]
df_big_c = pl.concat(list_big_c)

In [11]:
df_big_c_sel = df_big_c.select([
    pl.col("product_name").alias("name"),
    # strict=False จะเปลี่ยนทุกอย่างที่แปลงเป็น Float ไม่ได้ให้เป็น Null
    pl.col("sale_price").cast(pl.Float64, strict=False).alias("promotion_price"),
    pl.col("original_price").cast(pl.Float64, strict=False).alias("original_price")
])

In [12]:
# @title udf data-prep
def re_evaluate_price(df: pl.DataFrame) -> pl.DataFrame:
    """
    Standardizes pricing logic:
    1. If original_price is missing, move the promotion_price to it.
    2. If promotion_price matches the original_price, set it to Null.
    """
    return (
        df.with_columns(
            # Step 1: Fix missing original prices by 'swapping' from promotion_price
            pl.when(pl.col("original_price").is_null() & pl.col("promotion_price").is_not_null())
            .then(pl.col("promotion_price"))
            .otherwise(pl.col("original_price"))
            .alias("original_price")
        )
        .with_columns(
            # Step 2: Now that original_price is populated, nullify redundant promotions
            pl.when(pl.col("promotion_price") == pl.col("original_price"))
            .then(None)
            .otherwise(pl.col("promotion_price"))
            .alias("promotion_price")
        )
    )

# How to use it:
df_prep_big_c = re_evaluate_price(df_big_c_sel)

In [13]:
# @title udf Transform (Fixed Volume Extraction)
def parse_product_names(df: pl.DataFrame, shop_name: str) -> pl.DataFrame:
    """
    Pass any supermarket dataframe through this function to standardize the columns.
    Fixed to handle thousands separators (e.g., 1,300 ml).
    """
    # 1. Setup the dynamic date
    today_date = date.today().strftime("%Y-%m-%d")

    # 2. Updated patterns
    # Added [\d,.]+ to capture digits, commas, and dots
    quant_unit_pattern = r"(?i)([\d,.]+)\s*(ML|G|KG|L)\b"
    pack_pattern = r"(?i)(PACK\s*\d*\s*FREE\s*\d+|PACK\s*\d*\s*\+\s*\d+|PACK\s*\d+|TWINPACK|\bX\s*\d+\b|P?\d+\s*\+\s*\d+|\(\d+\+\d+\)|\d+\s*FREE\s*\d+|\bPACK\b)"

    # 3. Apply the Polars transformation
    return df.with_columns(
        pl.lit(today_date).alias("Date"),

        # Extract Brand
        pl.col("name").str.split(" ").list.first().alias("Brand"),

        # Fixed Volume Extraction:
        # 1. Extract the group (e.g., "1,300")
        # 2. Replace commas with nothing so it becomes "1300"
        # 3. Cast to Integer
        pl.col("name")
          .str.extract(quant_unit_pattern, 1)
          .str.replace_all(",", "")
          .cast(pl.Int64, strict=False)
          .alias("Volume"),

        # Extract Unit
        pl.col("name").str.extract(quant_unit_pattern, 2).str.to_uppercase().alias("Unit"),

        # Extract Pack size
        pl.col("name").str.extract(pack_pattern, 1).str.to_uppercase().alias("Pack"),

        # Add the dynamic Shop identifier
        pl.lit(shop_name).alias("Retailer")
    )

In [14]:
df_trans_big_c = parse_product_names(df_prep_big_c, "BigC")

In [15]:
df_trans_big_c

name,promotion_price,original_price,Date,Brand,Volume,Unit,Pack,Retailer
str,f64,f64,str,str,i64,str,str,str
"""FINELINE Fabric Softener Sunsh…",null,70.0,"""2026-04-09""","""FINELINE""",1300,"""ML""",null,"""BigC"""
"""FINELINE Fabric Softener Gentl…",null,70.0,"""2026-04-09""","""FINELINE""",1300,"""ML""",null,"""BigC"""
"""FINELINE Premium Clean Cleary …",null,179.0,"""2026-04-09""","""FINELINE""",1250,"""ML""",null,"""BigC"""
"""FINELINE Fabric Softener Pink …",null,70.0,"""2026-04-09""","""FINELINE""",1300,"""ML""",null,"""BigC"""
"""FINELINE Fabric Softener Red R…",null,70.0,"""2026-04-09""","""FINELINE""",1300,"""ML""",null,"""BigC"""
…,…,…,…,…,…,…,…,…
"""BIG C HAPPY PRICE Dish Washing…",null,25.0,"""2026-04-09""","""BIG""",400,"""ML""","""PACK""","""BigC"""
"""SUNLIGHT Plus Mind & Care Dish…",23.0,34.0,"""2026-04-09""","""SUNLIGHT""",500,"""ML""",null,"""BigC"""
"""MAXA Lemon Dishwashing Liquid …",null,58.0,"""2026-04-09""","""MAXA""",150,"""ML""","""PACK 6""","""BigC"""


In [16]:
df_trans_big_c.write_excel(f"big_c_laundry_dishwash_{today_date}.xlsx")